# 02 · Data Wrangling

**Stage 2 of 8** · ACCESS → WRANGLE → EXPLORE → VISUALIZE → ENGINEER → MODEL → EVALUATE → RECOMMEND

> **Project premise:** if someone has had cancer and was treated, **will it come back?** And for every factor linked to it coming back: **correlation or causation?**
>
> All data here is public: a k-anonymous feature table (every combination of site group, region, sex and diagnosis period is shared by ≥ 3 patients), synthetic patients, and aggregate reports. The raw database and clinical notes stay private.

**Goal of this stage:** find and fix the data-quality problems that would make the answer unreliable, and remove anything that identifies a person.

**Business question:** What data-quality issues could make the relapse rate, or the correlation-vs-causation verdict, wrong?

In [1]:
%matplotlib inline
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
PUBLIC = ROOT / "data" / "public" / "patients_features_public.csv"
SYNTH = ROOT / "data" / "synthetic" / "synthetic_patients.csv"
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "figure.dpi": 110})
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
df = pd.read_csv(PUBLIC)

The cleaning runs in `src/clean_anonymize.py` on the private raw dump. Below are its **aggregate**
results (from `reports/data_quality.json`), plus the cleaning functions demonstrated on **made-up** examples.

In [2]:
q = json.loads((ROOT / "reports" / "data_quality.json").read_text())
{k: v for k, v in q.items() if k not in ("tables", "region_counts")}

{'empty_tables': ['account', 'extrasummary', 'units'],
 'patients_raw_rows': 352,
 'test_records_removed': 1,
 'unique_patients_after_merge': 335,
 'rows_merged_or_dropped_as_duplicates': 16,
 'booking_numbers_reused_for_different_people': 1,
 'hospital_file_numbers_shared_by_rows': 6,
 'age_is_zero_share': 0.997,
 'rows_with_blank_diagnosis': 9,
 'rows_with_blank_summary': 17,
 'rows_with_blank_residence': 6,
 'ses_value_counts': {'II': 162, 'IV': 145, 'V': 29, '(blank)': 11, 'I': 5},
 'has_sex_column': False,
 'has_stage_column': False,
 'users_table_has_password_hash': True,
 'appointments_raw': 1270,
 'appointments_kept': 1250,
 'appointments_dropped_unlinkable': 20}

### Q1 · Which columns contain missing or unusable values?

In [3]:
pd.Series({
    "age = 0 (unusable)": f"{q['age_is_zero_share']:.1%} of rows",
    "blank diagnosis": q["rows_with_blank_diagnosis"],
    "blank clinical summary": q["rows_with_blank_summary"],
    "blank residence": q["rows_with_blank_residence"],
    "sex column exists?": q["has_sex_column"],
    "stage column exists?": q["has_stage_column"],
}, name="raw patients table").to_frame()

,raw patients table
age = 0 (unusable),99.7% of rows
blank diagnosis,9
blank clinical summary,17
blank residence,6
sex column exists?,False
stage column exists?,False


### Q2 · How do we handle missing values? (don't automatically delete)
- **Age** is 0 for every real patient → treated as *missing*, not as age 0. It can't be imputed honestly, so it is
  a stated limitation.
- **Sex** has no column → inferred from cancer site and pronouns in the notes; left `Unknown` when unclear.
- **Receptor status / grade** → `NaN` when the note doesn't say; we never fill it with a guess.
- In the public file, missingness itself is visible:

In [4]:
df[["sex", "grade", "er", "pr", "her2", "rfs_years"]].isna().sum().to_frame("missing")

,missing
sex,0
grade,242
er,278
pr,278
her2,280
rfs_years,197


### Q3 · Inconsistent categories: one residence, many spellings

In [5]:
from clean_anonymize import residence_to_region
examples = ["Khartoum - Omdurman", "khartum", "الخرطوم بحري", "Gezira - Madani", "نيالا", "Darfur - Fasher"]
pd.Series({e: residence_to_region(e) for e in examples}, name="region")

Khartoum - Omdurman                                         Khartoum
khartum                                                     Khartoum
الخرطوم بحري                                                Khartoum
Gezira - Madani        Central (Gezira / Sennar / White & Blue Nile)
نيالا                                                         Darfur
Darfur - Fasher                                               Darfur
Name: region, dtype: object

Free-typed residences in Arabic and English (with typos) are mapped to **7 regions**. This fixes the
inconsistency and also generalises a quasi-identifier (town → region).

### Q4 · Duplicate records: true duplicates or different people?

In [6]:
print(f"raw rows: {q['patients_raw_rows']}, test records removed: {q['test_records_removed']}")
print(f"merged duplicate entries: {q['rows_merged_or_dropped_as_duplicates']} → unique patients: {q['unique_patients_after_merge']}")
print(f"booking numbers reused for DIFFERENT people: {q['booking_numbers_reused_for_different_people']}")

raw rows: 352, test records removed: 1
merged duplicate entries: 16 → unique patients: 335
booking numbers reused for DIFFERENT people: 1


Duplicates were resolved by **hospital file number** or **booking number + same name**, never booking number
alone, because one booking number had been reused for two different people. Merging them blindly would have
mixed two patients' histories.

### Q5–Q6 · Data types and dates: free-text dates → a consistent year

In [7]:
from clean_anonymize import scrub_text
made_up = ["diagnosed 14/3/2016, operated Dec 2016, seen by Dr Smith",
           "on 2232018 started tamoxifen; ER 8/8 PR 0/8",
           "mastectomy in 5/2015 at Kassala"]
pd.DataFrame({"made-up note": made_up, "after cleaning": [scrub_text(t) for t in made_up]})

,made-up note,after cleaning
0,"diagnosed 14/3/2016, operated Dec 2016, seen b...","diagnosed 2016 , operated 2016, seen by [CLINI..."
1,on 2232018 started tamoxifen; ER 8/8 PR 0/8,on 2018 started tamoxifen; ER 8/8 PR 0/8
2,mastectomy in 5/2015 at Kassala,mastectomy in 2015 at [TOWN]


Dates come in many shapes (`dd/mm/yyyy`, glued `dmyyyy`, `Mon yyyy`). They are reduced to the **year**: enough
for time-to-relapse, and safer for privacy. Note that `ER 8/8` (a receptor score) is *not* mistaken for a date.

### Q7 · Suspicious values: the 'ses' column and the test record

In [8]:
pd.Series(q["ses_value_counts"], name="ses values").to_frame()

,ses values
II,162
IV,145
V,29
(blank),11
I,5


`ses` looks like a cancer stage (I, II, IV, V) but **isn't**: it doesn't line up with metastasis, palliative care or
site. It is kept as an unexplained code, and the clinic should be asked what it means. One record had a
joke diagnosis ("he's always happy") and was removed as test data. **Investigate before deleting**: here the
evidence was clear.

### Privacy: is the public file really anonymous? (k-anonymity validation)

In [9]:
qi = ["site_group", "region", "sex", "dx_period"]
sizes = df.groupby(qi, dropna=False).size()
print("smallest group sharing the same quasi-identifiers: k =", sizes.min())
print("region suppressed for", (df.region == "Suppressed").sum(), "patients to reach k ≥ 3")

smallest group sharing the same quasi-identifiers: k = 3
region suppressed for 128 patients to reach k ≥ 3


## So what?

The biggest quality issues aren't typos. They are **missing variables**: no age, no stage, no date of death, and receptor
status for only a third of breast patients. Those gaps, not the cleaning, are what limit the causal answer later.